# FCMFD Replay — SIA killing a candidate signal in seconds

This notebook replays an FCMFD-style SIA verdict using **genuinely synthetic data generated within this notebook**. It is NOT a copy of the real Phase 4 BTC/USDT or ETH/USDT data or Binance funding-rate data used in the original case study. The synthetic data is structured to reproduce the SIA verdict shape from the real case (Screen 2 lift-gap fails, Screen 4 vol-control dominates funding-control), so the reader sees the methodology in action with the same kind of output that produced the real verdict.

For the real case study, see [`../case-study/04-fcmfd-sia-kill-adr-0015.md`](../case-study/04-fcmfd-sia-kill-adr-0015.md).

## What this notebook demonstrates

1. **Synthetic data generation** — 2000 4-hour candles of a funding-rate-like series + volatility-like series + forward returns, with a weak funding-regime tilt deliberately inserted and a stronger vol-regime effect (the "signal carries information but it's too weak vs. the vol-control" failure shape).
2. **SIA orchestrator invocation** — the four screens running over the synthesised pooled frame via `sia.run_screens(pooled, jaccard_frame)`.
3. **Output JSON inspection** — what the screens report.
4. **Verdict interpretation** — how the FAIL routing maps to the kill decision and how to read each screen's evidence.

## Why this matters (in 3 seconds)

The SIA verdict you'll see at the bottom is computed in ~3 seconds. The alternative — running the same candidate through a 1000-epoch hyperopt × 2 seeds × 3 walk-forward windows — would take ~1 Hetzner-hour to reach the same conclusion. The four screens are doing the *cost-of-compute saving* that the framework exists for.


In [ ]:
import numpy as np
import pandas as pd

SEED = 42
N_CANDLES = 2000  # ~333 days at 4h cadence, comfortably above the gate's 97-day warm-up

rng = np.random.default_rng(SEED)
dates = pd.date_range("2024-01-01", periods=N_CANDLES, freq="4h")

# Latent volatility regime — slow oscillation so vol clusters by regime.
vol_phase = np.linspace(0.0, 8.0 * np.pi, N_CANDLES)
vol_regime = 0.015 + 0.010 * (0.5 + 0.5 * np.sin(vol_phase))  # ranges 0.015..0.025

# Latent funding signal — instantaneous component.
funding_signal = rng.normal(size=N_CANDLES)

# Funding *regime* (rolling mean of funding_signal) is what the gate picks up.
# Using the regime (not per-candle funding_signal) for the fwd-return tilt
# means the gate's smoothing doesn't destroy the visible lift in Screen 2.
funding_regime = pd.Series(funding_signal).rolling(42, min_periods=1).mean().to_numpy()

# Realised forward 10d return:
#   base drift
#   - 8.0  * (vol_regime - mean)   --> vol dominates Screen 4 (instantaneous vol_20d
#                                      observed ~1:1 vs vol_regime → tercile separation
#                                      is wide)
#   - 0.15 * funding_regime         --> small funding-regime tilt (Screen 2 lift is
#                                      positive but well below the 0.5σ bar; funding_4h
#                                      tercile in Screen 4 only weakly tracks fwd_ret
#                                      because funding_4h ≈ funding_signal, not
#                                      funding_regime)
#   + iid noise
fwd_ret_10d = (
    0.005
    - 8.0 * (vol_regime - vol_regime.mean())
    - 0.15 * funding_regime
    + 0.04 * rng.normal(size=N_CANDLES)
)
fwd_ret_5d = 0.5 * fwd_ret_10d + 0.025 * rng.normal(size=N_CANDLES)
fwd_ret_20d = 1.5 * fwd_ret_10d + 0.05 * rng.normal(size=N_CANDLES)

# Observed vol_20d (Screen 4's vol-control input) — regime + small noise.
vol_20d = vol_regime + 0.003 * rng.normal(size=N_CANDLES)

# Baseline filters (synthetic boolean masks).
macro_pass = rng.random(N_CANDLES) > 0.45     # ~55% trending regime
breakout_pass = rng.random(N_CANDLES) > 0.70  # ~30% breakouts

# Funding rate (Binance-style scale).
funding_4h = -0.0002 + 0.0008 * funding_signal

# Funding gates — replicate the production sia._internal.funding_gate logic
# inline so this notebook is self-contained. Pass: 7-day rolling-mean funding
# strictly below the 60th percentile of the trailing 90-day window.
funding_4h_series = pd.Series(funding_4h, index=dates)
SHORT_WINDOW = 42   # 7 days at 4h cadence
LONG_WINDOW = 540   # 90 days at 4h cadence
rolling_short_mean = funding_4h_series.rolling(SHORT_WINDOW, min_periods=SHORT_WINDOW).mean()
trailing = funding_4h_series.shift(SHORT_WINDOW)
trailing_p60 = trailing.rolling(LONG_WINDOW + 1, min_periods=LONG_WINDOW + 1).quantile(0.60)
trailing_p40 = trailing.rolling(LONG_WINDOW + 1, min_periods=LONG_WINDOW + 1).quantile(0.40)
funding_gate_pass = (rolling_short_mean < trailing_p60).fillna(False).to_numpy()
funding_gate_inversion = (~(rolling_short_mean < trailing_p40)).fillna(False).to_numpy()

pooled = pd.DataFrame(
    {
        "macro_pass": macro_pass,
        "breakout_pass": breakout_pass,
        "funding_gate_pass": funding_gate_pass,
        "funding_gate_inversion": funding_gate_inversion,
        "funding_4h": funding_4h,
        "vol_20d": vol_20d,
        "fwd_ret_5d": fwd_ret_5d,
        "fwd_ret_10d": fwd_ret_10d,
        "fwd_ret_20d": fwd_ret_20d,
        "pair": "SYNTH/USDT",
    },
    index=dates,
)

# Cohort diagnostics — preview the data before running the screens.
n_macro = int(pooled["macro_pass"].sum())
n_macro_breakout = int((pooled["macro_pass"] & pooled["breakout_pass"]).sum())
n_treatment = int((pooled["macro_pass"] & pooled["breakout_pass"] & pooled["funding_gate_pass"]).sum())
n_inversion = int((pooled["macro_pass"] & pooled["breakout_pass"] & pooled["funding_gate_inversion"]).sum())
print(f"Generated {len(pooled)} synthetic 4h candles spanning {pooled.index.min()} to {pooled.index.max()}.\n")
print(f"  macro_pass             : {n_macro:>4} ({n_macro / len(pooled):.1%})")
print(f"  + breakout_pass        : {n_macro_breakout:>4}  (control_b cohort)")
print(f"  + funding_gate_pass    : {n_treatment:>4}  (treatment cohort)")
print(f"  + funding_gate_inversion: {n_inversion:>4}  (inversion cohort)")


## Run the four SIA screens

`sia.run_screens` is the library-API entry point. It takes the pooled per-candle dataframe (with the boolean and forward-return columns the four screens read) plus a `jaccard_frame` for Screen 3's hyperparameter-grid sweep. In a real multi-pair pipeline `jaccard_frame` is one per-pair frame; for this single-pair synthetic demo we reuse `pooled`.

The return is a dict with the top-level verdict (`PASS` / `FAIL` / `FAIL+INVERT`), the exit code (0 / 1 / 2), and per-screen evidence dicts. Read the verdict first, then dig into evidence for the screens that failed.


In [ ]:
import json

import sia

verdict = sia.run_screens(pooled, jaccard_frame=pooled)

print(f"Overall verdict: {verdict['verdict']}  (exit code {verdict['exit_code']})\n")
for screen_key, screen_data in verdict["evidence"].items():
    print(f"  {screen_key:<35} {screen_data['verdict']:<6}  {screen_data['notes']}")

print("\n--- Full evidence (JSON) ---")
print(json.dumps(verdict, indent=2, default=str))


## Interpret the verdict

Walk through the four screens in order.

### Screen 1 — Coverage (PASS)

The new gate has to do *something* operationally: it must reduce the candidate-trade frequency vs the macro+breakout baseline by ≥ 20%, and the conjunction must yield ≥ 30 trades. Both are mechanical hygiene checks — they rule out "the gate is a no-op" and "the gate is so strict nothing trades."

With the synthetic data above, the gate cuts the macro+breakout cohort from 314 to 228 candles (≈ 27% reduction, well above the 20% floor) and gives 228 trades (well above the 30 floor). The gate is doing real work.

### Screen 2 — Lift (FAIL)

The locked direction (funding-gated cohort vs price-only macro+breakout) has to beat the control by ≥ 0.5 * pooled-stddev on at least 2 of 3 horizons (5d, 10d, 20d). With the synthetic data:

| Horizon | Treatment mean | Control mean | Gap σ | Threshold | Verdict |
| ------- | -------------- | ------------ | ----- | --------- | ------- |
| 5d      | small positive | small positive | ~0.045 | 0.5 | FAIL |
| 10d     | small positive | small positive | ~0.075 | 0.5 | FAIL |
| 20d     | small positive | small positive | ~0.056 | 0.5 | FAIL |

The treatment cohort outperforms the control on every horizon — the funding-regime tilt is real and qualitatively in the right direction — but the magnitudes are far below the pre-committed 0.5σ bar. Cohort variance swamps the lift.

This is what the real FCMFD case looked like too: treatment mean 2.97% vs control 1.86% at 5d (gap σ 0.147), treatment 3.66% vs control 2.44% at 10d (gap σ 0.098), treatment 3.99% vs control 3.69% at 20d (gap σ 0.016). The case-study writeup makes the same diagnosis: directionally confirmed, magnitude too small to bank on.

### Screen 3 — Jaccard (PASS)

The 5×5 grid over `(atr_mult, funding_threshold_pct)` produces 300 pairwise entry-set comparisons. Mean Jaccard ~0.67, comfortably below the 0.85 threshold — different threshold values do produce different entry sets, so the signal isn't trivially encoded in a hyperparameter knob.

### Screen 4 — Information split (FAIL)

This is the sharpest diagnosis. We bin the macro+breakout candidates two ways — funding-tercile and vol-tercile — and measure tercile separation on each metric (mean, win rate, Sharpe). The funding binning needs to show wider separation than the vol binning on ≥ 2 of 3 metrics. It doesn't:

| Metric  | Funding separation | Vol separation | Wider |
| ------- | ------------------ | -------------- | ----- |
| mean    | ~0.015             | ~0.043          | vol  |
| winrate | ~0.17              | ~0.40           | vol  |
| Sharpe  | ~0.27              | ~0.95           | vol  |

Vol-tercile separation is ~2–3× wider on every metric. The funding signal *carries information* — the tercile means are still monotone in the right direction — but realised volatility already explains more of the forward-return variance. A vol-aware baseline would absorb most of funding's signal anyway, which is the whole point of the information-split control.

### Why the inversion clause did not fire

ADR-0014's inversion clause reserves a "wrong-direction" retest path for the case where (a) the locked direction fails Screen 2 *and* (b) the inverted-direction cohort shows ≥ 1.0σ separation on ≥ 2 of 3 horizons. The locked direction failed Screen 2 (✓), but the inversion cohort underperformed control on every horizon with small *negative* gap σ (~-0.083 / -0.138 / -0.111). The inversion clause is not authorised. No Phase-4-prime authorisation; the candidate is killed without retest.

### What you'd do next

The case-study writeup at `../case-study/04-fcmfd-sia-kill-adr-0015.md` lists the next-step options (different asset class, different timeframe, different baseline, or accept and move on). The point of the framework is that you reach this fork at compute-cost ≈ 0 instead of compute-cost ≈ Hetzner-hour-per-candidate.
